# Module 4: Saturate Your GPU

In Module 3 you served the model with [vLLM](https://docs.vllm.ai) and watched continuous batching turn many concurrent requests into one efficient pass over the GPU. Batching helps, but how far does it go, and where does it break? This module finds out the only honest way: you drive rising concurrency at your server until throughput stops climbing and time to first token starts to. You will watch the server pass through five stages, and read the one metric that proves each one, so the bottleneck is something you see in the numbers rather than something you guess at. The server under load is the same one from Module 3, served on an [Akamai Cloud GPU](https://www.linode.com/products/gpu/).

## Learning objectives
- Drive a vLLM endpoint at rising concurrency with a repeatable load generator
- Watch throughput climb, flatten at the knee, then stop paying off
- Read `num_requests_waiting` to see the queue back up before the hardware does
- Read `kv_cache_usage_perc` and `num_preemptions_total` to catch the server protecting itself
- Plot throughput and TTFT against concurrency and pick the knee
- Name the bottleneck (batch cap, queue, KV cache, or compute) from the metrics, not a hunch

## Prerequisites
- Finished Module 3, with vLLM serving and `/metrics` reachable
- A live vLLM endpoint in `VLLM_HOST`, and its `/metrics` URL resolving from `get_settings()`
- You know what `num_requests_running` and `kv_cache_usage_perc` mean from Module 2
- About 15 minutes

References: [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [vLLM benchmarking CLI](https://docs.vllm.ai/en/latest/benchmarking/cli/) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/serving/metrics.html) &middot; [Continuous batching](https://www.anyscale.com/blog/continuous-batching-llm-inference)

## Saturation design basics

Raising concurrency does not break a server all at once. It breaks in order, and each stage leaves a fingerprint in a different metric. Watching the right gauge tells you which stage you are in.

- **Headroom.** The batch grows, throughput climbs, latency barely moves. The GPU is filling up but not full. The proof is rising `output_throughput`.
- **The knee.** Throughput flattens. The batch is already full, so adding concurrent requests stops adding tokens per second. Per-request latency starts rising. This is the operating point you care about.
- **Queue.** New requests cannot get a batch slot, so `num_requests_waiting` grows. TTFT climbs because requests wait before prefill even starts.
- **KV cache pressure.** `kv_cache_usage_perc` approaches 1.0. There is no room left to store attention state for more tokens.
- **Preemption.** vLLM evicts a running request to free KV blocks, then recomputes it later. `num_preemptions_total` starts climbing. This is the server protecting itself, and it is the clearest sign you are past the limit.

On a small model and a large card, the binding limit is often the batch cap and the queue, not KV memory. Watch `num_requests_waiting`: it can back up while the KV cache is still half empty. You will see exactly which signal hits first.

![Five saturation stages left to right, headroom then knee then queue then KV pressure then preemption, each labeled with the metric that proves it](images/04_saturate_your_gpu_architecture.png)

One law names what the plot shows. By Little's Law, the requests inside the server equal the arrival rate times the time each one spends there: `L = lambda * W`. Below the knee, more concurrency raises throughput while latency holds, so you get real work for the load. At the knee the service rate is maxed, so added load cannot raise throughput, it only grows `W`: the queue (`num_requests_waiting`) climbs and TTFT goes hockey-stick. The roofline below says why the knee sits where it does on your card.

![A roofline with a sloped memory-bandwidth ceiling and a flat compute ceiling meeting at the ridge point. Decode at batch 1 sits far down the memory-bound slope, batching walks it up toward the ridge, and prefill already sits near the top.](images/04_roofline_ridge_point.png)

## 1. Setup

This module reads its connection details from `common/config.py` and samples the live `/metrics` endpoint with `common/metrics.py`. Install the plotting dependency this module needs. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q matplotlib

## 2. Configure endpoint, metrics, and load tool

`get_settings()` resolves `VLLM_HOST`, `MODEL_NAME`, and the `/metrics` URL. The load generator needs the server root without `/v1`, plus an explicit endpoint path, so we strip `/v1` here. `vllm bench serve` ships with the hosted image; when the CLI is missing (for example running against a remote endpoint from a laptop), the sweep falls back to the pure-Python load generator in `common/load.py` automatically.

In [ ]:
# Setup: settings, the metrics URL, and the server root for the load tool.
import os, sys, json, time, shutil, subprocess, threading, tempfile
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings
from common import metrics, load

settings = get_settings()

# vllm bench serve wants the server root (no /v1) plus an explicit endpoint path.
server_root = settings.vllm_host.rstrip("/")
if server_root.endswith("/v1"):
    server_root = server_root[:-3]

# The hosted image ships the `vllm` CLI. When it is missing (for example
# running against a remote endpoint from a laptop), the sweep falls back to the
# pure-Python load generator in common/load.py automatically.
HAVE_VLLM_CLI = shutil.which("vllm") is not None

print("server root :", server_root)
print("metrics URL :", settings.metrics_url)
print("model       :", settings.model_name)
print("load tool   :", "vllm bench serve" if HAVE_VLLM_CLI else "pure-Python fallback (common/load.py)")

**What you should see:** your server root (the base URL without `/v1`), the metrics URL, and the model. The load tool prints which generator it will use: the `vllm` CLI if present, otherwise the pure-Python fallback. Either one drives the same sweep.

## 3. Define the per-level load runner

`vllm bench serve` fires a dataset of prompts at your endpoint at a chosen concurrency and reports throughput, TTFT, and TPOT. You run it once per concurrency level and collect the numbers. The helper below runs one level, samples `/metrics` in the background to capture the peak KV cache and waiting count, reads the preemption counter before and after, and returns everything in one dict. When the CLI is absent it hands off to `load.run_level`, which returns the same fields.

In [ ]:
# Requires a live vLLM endpoint. Uses the vllm CLI when present, otherwise the
# pure-Python load generator in common/load.py (same fields either way).
# Run one concurrency level: load test + metric sampling, return a result dict.
def run_level(concurrency, num_prompts=None, input_len=256, output_len=128):
    if not HAVE_VLLM_CLI:
        from common.config import build_client
        return load.run_level(
            build_client(settings), settings.model_name, settings.metrics_url,
            concurrency, num_prompts=num_prompts, output_len=output_len,
        )

    num_prompts = num_prompts or max(concurrency * 4, 16)

    # Sample preemptions and KV cache in the background during the run.
    peak = {"kv": 0.0, "waiting": 0.0}
    preempt_start = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]
    stop = threading.Event()

    def sample():
        while not stop.is_set():
            try:
                s = metrics.snapshot(settings.metrics_url)
                peak["kv"] = max(peak["kv"], s["vllm:gpu_cache_usage_perc"])
                peak["waiting"] = max(peak["waiting"], s["vllm:num_requests_waiting"])
            except Exception:
                pass
            time.sleep(0.25)

    t = threading.Thread(target=sample, daemon=True)
    t.start()

    out_path = os.path.join(tempfile.gettempdir(), f"bench_c{concurrency}.json")
    cmd = [
        "vllm", "bench", "serve",
        "--backend", "openai-chat",
        "--base-url", server_root,
        "--endpoint", "/v1/chat/completions",
        "--model", settings.model_name,
        "--dataset-name", "random",
        "--random-input-len", str(input_len),
        "--random-output-len", str(output_len),
        "--num-prompts", str(num_prompts),
        "--max-concurrency", str(concurrency),
        "--save-result", "--result-filename", out_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    stop.set(); t.join(timeout=2)

    if proc.returncode != 0:
        print(proc.stdout[-500:]); print(proc.stderr[-500:])
        raise RuntimeError(f"vllm bench serve failed at concurrency {concurrency}")

    with open(out_path) as f:
        data = json.load(f)
    preempt_end = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]

    return {
        "concurrency": concurrency,
        "output_throughput": data.get("output_throughput"),       # tokens/s
        "request_throughput": data.get("request_throughput"),     # req/s
        "mean_ttft_ms": data.get("mean_ttft_ms"),
        "mean_tpot_ms": data.get("mean_tpot_ms"),
        "peak_kv": peak["kv"],
        "peak_waiting": peak["waiting"],
        "preemptions": preempt_end - preempt_start,
    }

**What you should see:** nothing yet. This defines the per-level runner. Each call takes the better part of a minute, so the sweep below runs only a handful of levels.

## 4. Sweep rising concurrency

Run the sweep from gentle to brutal. Watch the printed line for each level: throughput should climb, then flatten, while TTFT and the waiting count grow and preemptions appear. That transition, from one line to the next, is the bottleneck showing itself on screen.

In [ ]:
# Requires a live vLLM endpoint.
# Sweep concurrency from 1 to 128 and collect a result row per level.
levels = [1, 4, 16, 64, 128]
results = []
for c in levels:
    r = run_level(c)
    results.append(r)
    print(
        f"c={c:>3}  out={r['output_throughput']:>7.0f} tok/s  "
        f"TTFT={r['mean_ttft_ms']:>7.0f} ms  TPOT={r['mean_tpot_ms']:>5.1f} ms  "
        f"KV={r['peak_kv']*100:>4.0f}%  wait={r['peak_waiting']:>4.0f}  "
        f"preempt={r['preemptions']:>4.0f}"
    )

**What you should see:** output throughput rising from `c=1` through the middle of the sweep, then flattening or dipping at the top. TTFT and the waiting count grow as you climb. On a small model and a large card, watch the `wait` column climb while `KV` may still be well under 100 percent: the queue, not the KV cache, can be your first wall. The exact numbers depend on your model and GPU.

## 5. Plot the knee

Two curves tell the story: throughput against concurrency, which bends over at the knee, and TTFT against concurrency, which turns up where queueing starts. Read them together to pick the highest concurrency that still holds your latency target.

In [ ]:
# Plot throughput and TTFT against concurrency to find the knee.
import matplotlib.pyplot as plt

xs = [r["concurrency"] for r in results]
thru = [r["output_throughput"] for r in results]
ttft = [r["mean_ttft_ms"] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(xs, thru, marker="o")
ax1.set_xlabel("concurrency"); ax1.set_ylabel("output tokens/s")
ax1.set_title("Throughput vs concurrency"); ax1.set_xscale("log", base=2); ax1.grid(True, alpha=0.3)

ax2.plot(xs, ttft, marker="o", color="tab:red")
ax2.set_xlabel("concurrency"); ax2.set_ylabel("mean TTFT (ms)")
ax2.set_title("TTFT vs concurrency"); ax2.set_xscale("log", base=2); ax2.grid(True, alpha=0.3)
fig.tight_layout()

**What you should see:** a throughput curve that rises then flattens, and a TTFT curve that stays low then turns sharply up. The concurrency where throughput stops climbing but latency starts climbing is the knee. That is your honest capacity on the current configuration.

## 6. Name the bottleneck

Look at the level where throughput flattened, then ask which signal hit its limit first:

- If `peak_kv` reached ~100 percent and `preemptions` jumped, you are **KV cache bound**. The GPU ran out of room to store attention state for concurrent requests. The next module fixes this by giving vLLM more of the GPU for the cache.
- If `peak_waiting` climbed hard while `peak_kv` stayed well under 100 percent, you are **batch-cap or queue bound**. Requests are waiting for a slot, not for memory. This is the common case for a small model on a large card, and the next module fixes it by raising `max-num-seqs`.
- If throughput flattened with both KV and the queue calm, you are **compute bound** on prefill or decode. More concurrency cannot help; a faster GPU, a smaller model, or quantization would.

Write down which one you saw. The next module changes the engine flags to move it.

In [ ]:
# Print the level where throughput peaked and the signals at the top of the sweep.
peak_level = max(results, key=lambda r: r["output_throughput"] or 0)
top = results[-1]
print(f"throughput peaked at concurrency {peak_level['concurrency']} "
      f"({peak_level['output_throughput']:.0f} tok/s)")
print(f"at the top (c={top['concurrency']}): "
      f"KV={top['peak_kv']*100:.0f}%, waiting={top['peak_waiting']:.0f}, "
      f"preemptions={top['preemptions']:.0f}")
verdict = "KV cache bound" if top["peak_kv"] > 0.95 and top["preemptions"] > 0 else "compute or queue bound (see notes above)"
print("likely bottleneck:", verdict)

**What you should see:** the concurrency at which throughput peaked, the state of the server at the top of the sweep, and a first-guess verdict. Confirm it against the plots and the per-level table. If `waiting` is high while `KV` is low, your wall is the batch cap, not memory.

### Why the knee sits where it does

The verdict above came from the metrics. Here is the physics behind it. Every GPU has a ridge point, the arithmetic intensity (`FLOPs per byte`) where it stops being memory-bound and turns compute-bound: `ridge = peak_compute / memory_bandwidth`. A single decode step does very little compute per byte it reads, so it sits far below the ridge, deep in memory-bound territory. Batching raises the work done per byte and walks you toward the ridge. Compute the ridge for your card and see how little of it one request uses.

In [ ]:
# The ridge point is where a GPU flips from memory-bound to compute-bound.
# Fill in two numbers from your card's datasheet.
peak_fp16_tflops = 150.0   # dense fp16 compute, TFLOP/s
bandwidth_tbs    = 0.36    # memory bandwidth, TB/s

ridge = peak_fp16_tflops / bandwidth_tbs   # TFLOP/s over TB/s is FLOPs per byte
print(f"ridge point: {ridge:.0f} FLOPs/byte  (below this you are memory-bound)")

# A batch of B decode requests reuses one weight load for B tokens, so its
# arithmetic intensity is roughly B FLOPs/byte. Compare that to the ridge.
for B in [1, 16, 64, 128]:
    intensity  = 1.0 * B
    efficiency = min(1.0, intensity / ridge)
    print(f"batch {B:>3}: ~{intensity:>3.0f} FLOPs/byte  ->  ~{efficiency*100:>4.1f}% of compute used")

**What you should see:** a ridge point in the hundreds of FLOPs per byte and a decode efficiency in the low single digits at small batch, climbing as the batch grows. This is the calculation behind the verdict: when efficiency is low, KV cache is low, and `waiting` is high, you are queue-bound, not compute-bound, the common case for a small model on a large card. Raising the batch cap in Module 5 is what walks you up this curve.

## 7. Fallback: pure-Python load (no vllm CLI)

If `vllm bench serve` is not available, this generates load with the OpenAI client and a thread pool. It is cruder (no token-length control, simpler stats), but it still drives concurrency high enough to show the knee and trigger preemption. The sweep in Section 4 already dispatches to `common/load.py` for you; run this cell only if you want to drive the raw client by hand.

In [ ]:
# Requires a live vLLM endpoint. Use only if the vllm CLI is unavailable.
# Drive concurrency with a thread pool and read peak KV + preemptions from /metrics.
from concurrent.futures import ThreadPoolExecutor
from common.config import build_client

client = build_client(settings)

def fire_one():
    t = time.time()
    r = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": "Write a detailed paragraph about GPU memory."}],
        max_tokens=128, temperature=0.0,
    )
    return r.usage.completion_tokens, time.time() - t

def py_load(concurrency, total=None):
    total = total or concurrency * 4
    p0 = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]
    peak_kv = 0.0
    stop = threading.Event()
    def s():
        nonlocal peak_kv
        while not stop.is_set():
            try: peak_kv = max(peak_kv, metrics.snapshot(settings.metrics_url)["vllm:gpu_cache_usage_perc"])
            except Exception: pass
            time.sleep(0.25)
    th = threading.Thread(target=s, daemon=True); th.start()
    start = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        res = list(pool.map(lambda _: fire_one(), range(total)))
    wall = time.time() - start
    stop.set(); th.join(timeout=2)
    toks = sum(t for t, _ in res)
    p1 = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]
    print(f"c={concurrency:>3}  {toks/wall:>7.0f} tok/s  peakKV={peak_kv*100:>4.0f}%  preempt={p1-p0:>4.0f}")

for c in [1, 8, 32, 96]:
    py_load(c)

**What you should see:** the same shape as the CLI sweep. Throughput climbs then flattens, peak KV cache rises toward 100 percent, and preemptions appear at the high end. Less precise than `vllm bench serve`, but the bottleneck still shows up.

## Things to know

- **The knee is the operating point.** Past it you pay latency for no extra throughput. Pick the highest concurrency that still holds your TTFT target; that is your honest capacity, not the peak throughput number alone.
- **Watch the queue first on a small model.** On a 4B model and a 20GB card, the binding limit is usually the batch cap, so `num_requests_waiting` backs up while the KV cache is still half empty. The metric that hits its wall first names your bottleneck.
- **Preemption is recompute, not failure.** vLLM's V1 engine evicts a running request and recomputes it later to make room. `num_preemptions_total` climbing is the server protecting itself; it is a signal you are over the limit, not a crash.
- **The KV gauge was renamed.** vLLM's V1 engine exposes the KV cache fraction as `kv_cache_usage_perc`; older builds called it `gpu_cache_usage_perc`. The `snapshot()` helper returns the value under both names, so reads work whichever your server emits.
- **The sweep is destructive on purpose.** It drives your server past its limit, so run it against a test endpoint, not one serving live traffic.
- **Little's Law names the knee.** The requests in the server equal arrival rate times time in the server, `L = lambda * W`. Past the knee the service rate is fixed, so more load cannot add throughput, it only grows the wait. `num_requests_waiting` rising above zero is that law showing up in a metric: you have passed the knee.

> NOTE: Each load level takes the better part of a minute. Keep `levels` short while you iterate; widen it once the shape is clear.

## Try it yourself

**Find your exact knee.** Add finer levels between the two where throughput flattened (for example `8, 12, 24, 32`) and re-run to locate the concurrency where the curve bends. That number is the capacity you quote. **Stretch:** print throughput per unit of TTFT at each level and pick the level that maximizes it.

**Push past KV cache.** Raise `output_len` in `run_level` so each request holds more tokens in the cache, then re-run one high level. Watch `peak_kv` climb faster and preemptions arrive sooner: longer outputs reach KV pressure at lower concurrency.

**Make the queue the wall.** Lower `output_len` and raise the top concurrency. With short outputs the KV cache stays calm, so `peak_waiting` should be the first signal to spike. Confirm the batch cap, not memory, is your limit.

In [ ]:
# Change one level and the output length, then run the cell.
test_concurrency = 64      # a single level to probe
test_output_len = 256      # raise this to push KV cache, lower it to push the queue

r = run_level(test_concurrency, output_len=test_output_len)
print(
    f"c={r['concurrency']}  out={r['output_throughput']:.0f} tok/s  "
    f"TTFT={r['mean_ttft_ms']:.0f} ms  "
    f"KV={r['peak_kv']*100:.0f}%  wait={r['peak_waiting']:.0f}  "
    f"preempt={r['preemptions']:.0f}"
)

## Summary

- Batching helps until the batch is full. You drove rising concurrency and watched throughput climb, hit the knee, then stop paying off.
- Saturation arrives in stages, and each leaves a fingerprint: `num_requests_waiting` for the queue, `kv_cache_usage_perc` for memory, `num_preemptions_total` for the server protecting itself.
- The throughput-versus-TTFT knee is your operating point. The plot finds it; the metrics name the bottleneck behind it.
- On a small model and a large card, the wall is usually the batch cap and the queue, not KV memory. The metric that hits its limit first decides which flag to change next.

## Next

**Module 5: Optimize the Server.** You can name the bottleneck now, so next you change the engine flags that move it. You will raise `max-num-seqs` and `gpu-memory-utilization`, redeploy, and re-run this exact sweep to prove the knee moved.